# LegalQA v9.2 — SFT LoRA thuần cho Qwen3.5-2B trên `train_sft.json`
Dùng đúng full pipeline RAG thật lúc inference (BM25+dense+reranker qua `retrieve_sqlite_batch`, backend SQLite) để build context cho từng câu train — không train trên câu hỏi trần, không dùng weak-label alignment. Mỗi epoch được lưu checkpoint riêng; sau training, script sinh câu trả lời trên dev cho từng checkpoint và chọn checkpoint có METEOR cao nhất — **không chọn theo eval_loss/training loss**, vì loss giảm không đồng nghĩa METEOR trên câu sinh ra tăng.

METEOR/ROUGE-L tính đúng công thức `Scoring-Program-Task-LegalQA/scoring.py` của BTC (nltk `meteor_score` có WordNet + `rouge_score` RougeL thật, không phải probe xấp xỉ) nên số đo được ở đây bám sát điểm leaderboard. Dependency `nltk`/`rouge-score`/corpus `wordnet`+`omw-1.4` nằm trong `wheelhouse_scoring/` (tách riêng khỏi `wheelhouse/` chính để không phá invariant 12-wheel của 3 notebook v9.2 khác) và được cell dưới cài + trỏ `NLTK_DATA` tự động — không cần internet trên Kaggle.

Input cần attach trên Kaggle: source v9.2 (đã pass `SOURCE_CONTRACT.json`, gồm cả `wheelhouse_scoring/`), checkpoint Qwen3.5-2B approved + embedding + reranker, corpus (`selected-contexts`), một global dense index đã build sẵn (`manifest.json` schema `global-dense-v1`, ví dụ output của `run_inference_only.py --dense-index-only`), và một dataset chứa `train_sft.json` (từ `scripts/preprocess_train_sft.py`). Nếu có sẵn `dev_sft.json` riêng thì đính kèm luôn để dùng làm dev cố định; không có thì notebook tự tách dev theo `DEV_RATIO`. Notebook tự build SQLite corpus store + chạy retrieval thật (không cần attach `sft_evidence.jsonl` build sẵn).

In [ ]:
EPOCHS = 5.0                # lưu checkpoint sau mỗi epoch; xem cell cuối để chọn epoch tốt nhất theo METEOR dev
LEARNING_RATE = 1e-4
GRAD_ACCUM = 16
MAX_LENGTH = 3072
MAX_TARGET_TOKENS = 700
MAX_CONTEXT_CHARS = 3500       # ngân sách ký tự evidence riêng cho SFT (đo lại bằng tokenizer thật trước khi chốt)
LORA_R = 8
DEV_RATIO = 0.15            # dùng khi KHÔNG đính kèm dev_sft.json riêng
MAX_TRAIN_SAMPLES = None    # đặt vd 100 để smoke test nhanh
MAX_EVAL_SAMPLES = None     # giới hạn số câu dev khi chọn epoch để sinh nhanh hơn (vd 100)
AUTO_RESUME_TRAINING = True

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
input_root = Path('/kaggle/input')
EXPECTED_SOURCE_CONTRACT = {
    'schema': 'legalqa-source-contract-v1',
    'release': '0.9.2',
    'agentic_namespace': 'agentic-v3',
    'cache_schema': 'inference-v7-agentic-error-isolation',
    'selector_schema': 'selector-weights-v3-agentic-evidence',
    'offline_wheel_count': 12,
}
REQUIRED_WHEELS = ('transformers-', 'tokenizers-', 'huggingface_hub-', 'hf_xet-', 'safetensors-', 'accelerate-', 'peft-', 'bitsandbytes-', 'sentence_transformers-', 'sentencepiece-', 'regex-', 'click-')
REQUIRED_SCORING_WHEELS = ('nltk-', 'rouge_score-', 'joblib-', 'absl_py-')
def source_contract_errors(path):
    errors = []
    marker = path / 'SOURCE_CONTRACT.json'
    try:
        contract = json.loads(marker.read_text(encoding='utf-8'))
    except Exception as exc:
        return [f'thiếu/hỏng SOURCE_CONTRACT.json: {exc}']
    for key, expected in EXPECTED_SOURCE_CONTRACT.items():
        if contract.get(key) != expected:
            errors.append(f'{key}={contract.get(key)!r}, cần {expected!r}')
    required_files = (
        path / 'kaggle' / 'run_qwen_sft_lora.py',
        path / 'scripts' / 'train_qwen_sft_lora.py',
        path / 'scripts' / 'select_best_sft_checkpoint.py',
        path / 'RELEASE_V9_2.md',
        path / 'wheelhouse_scoring' / 'nltk_data' / 'corpora' / 'wordnet' / 'lexnames',
        path / 'wheelhouse_scoring' / 'nltk_data' / 'corpora' / 'omw-1.4' / 'citation.bib',
    )
    errors.extend(f'thiếu {file.relative_to(path)}' for file in required_files if not file.is_file())
    wheels = list((path / 'wheelhouse').glob('*.whl'))
    names = [wheel.name for wheel in wheels]
    missing_wheels = [prefix for prefix in REQUIRED_WHEELS if not any(name.startswith(prefix) for name in names)]
    if len(wheels) != 12 or missing_wheels or any(wheel.stat().st_size < 10_000 for wheel in wheels):
        errors.append(f'wheelhouse không hợp lệ: count={len(wheels)}, missing={missing_wheels}')
    scoring_wheels = list((path / 'wheelhouse_scoring').glob('*.whl'))
    scoring_names = [wheel.name for wheel in scoring_wheels]
    missing_scoring_wheels = [prefix for prefix in REQUIRED_SCORING_WHEELS if not any(name.startswith(prefix) for name in scoring_names)]
    if missing_scoring_wheels:
        errors.append(f'wheelhouse_scoring thiếu wheel: {missing_scoring_wheels}')
    return errors
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml')})
source_reports = {str(path): source_contract_errors(path) for path in source_projects}
qualified_sources = [path for path in source_projects if not source_reports[str(path)]]
if len(qualified_sources) != 1:
    raise RuntimeError(f'Cần đúng một source v9.2 reviewed; source cũ nhúng trong output cũ sẽ bị loại. qualified={qualified_sources}; reports={source_reports}')
selected_source = qualified_sources[0]
print('Source v9.2 reviewed:', selected_source)
print('Rejected source copies:', {path: reasons for path, reasons in source_reports.items() if reasons})
project = Path('/kaggle/working/legalqa_source')
if project.exists():
    shutil.rmtree(project)
shutil.copytree(selected_source, project)
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
scoring_wheels = sorted(str(path) for path in (project / 'wheelhouse_scoring').glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *scoring_wheels], check=True)
os.environ['PYTHONPATH'] = str(project / 'src')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['NLTK_DATA'] = str(project / 'wheelhouse_scoring' / 'nltk_data')
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], check=True)
print('Source:', selected_source)

In [ ]:
resume_roots = []
if AUTO_RESUME_TRAINING:
    for signature in input_root.rglob('resume_signature.json'):
        if any(p.is_dir() and p.name.startswith('checkpoint-') for p in signature.parent.iterdir()):
            resume_roots.append(signature.parent)
resume_roots = sorted(set(resume_roots))
if len(resume_roots) > 1:
    raise RuntimeError(f'Chỉ attach một SFT training state phù hợp: {resume_roots}')
command = [
    sys.executable, str(project / 'kaggle' / 'run_qwen_sft_lora.py'),
    '--config', str(project / 'configs' / 'kaggle_inference_t4.yaml'),
    '--input-root', '/kaggle/input',
    '--working', '/kaggle/working/legalqa_sft',
    '--dev-ratio', str(DEV_RATIO),
    '--epochs', str(EPOCHS),
    '--learning-rate', str(LEARNING_RATE),
    '--grad-accum', str(GRAD_ACCUM),
    '--max-length', str(MAX_LENGTH),
    '--max-target-tokens', str(MAX_TARGET_TOKENS),
    '--max-context-chars', str(MAX_CONTEXT_CHARS),
    '--lora-r', str(LORA_R),
]
if MAX_TRAIN_SAMPLES is not None:
    command.extend(['--max-train-samples', str(MAX_TRAIN_SAMPLES)])
if MAX_EVAL_SAMPLES is not None:
    command.extend(['--max-eval-samples', str(MAX_EVAL_SAMPLES)])
if resume_roots:
    command.extend(['--resume-input', str(resume_roots[0])])
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
root = Path('/kaggle/working/legalqa_sft')
manifest_path = root / 'sft_manifest.json'
assert manifest_path.exists(), 'Training chưa hoàn tất. Hãy Save Version để giữ checkpoint rồi attach lại để resume.'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('Epoch -> METEOR / ROUGE-L trên dev (đúng công thức chấm của BTC):')
for row in manifest['epoch_selection']['results']:
    print(f"  epoch={row['epoch']:.2f}  meteor={row['meteor']:.4f}  rouge_l_f1={row['rouge_l_f1']:.4f}")
best = manifest['epoch_selection']['best']
print('Checkpoint tốt nhất theo METEOR dev:', best)
adapter_zip = Path(manifest['adapter_zip'])
assert adapter_zip.is_file()
print('Adapter zip:', adapter_zip)

# Output đầy đủ model sinh ra trên dev + train của checkpoint tốt nhất — lưu nguyên trong
# /kaggle/working/legalqa_sft/qwen-sft-lora/predictions_{dev,train}_checkpoint-*.json,
# Save Version để giữ lại đọc sau (không mất khi kernel tắt).
for split_key, label in (('dev_predictions_file', 'DEV'), ('train_predictions_file', 'TRAIN')):
    path_str = best.get(split_key)
    if not path_str:
        print(f"\n(không có {label.lower()} predictions file)")
        continue
    predictions_path = Path(path_str)
    predictions = json.loads(predictions_path.read_text(encoding='utf-8'))
    print(f"\n[{label}] {predictions_path} — {len(predictions)} câu, xem 10 mẫu đầu:")
    for question_id, pair in list(predictions.items())[:10]:
        print(f"--- qid={question_id} ---")
        print('ref :', pair['reference'][:500])
        print('pred:', pair['prediction'][:500])

print(json.dumps(manifest, ensure_ascii=False, indent=2))